# cell_sim on Colab — JCVI-Syn3A event-driven simulator

Multi-scale event-driven simulation of the JCVI-Syn3A minimal cell, built on real kinetic data from the [Luthey-Schulten Lab](https://github.com/Luthey-Schulten-Lab/Minimal_Cell_ComplexFormation).

This notebook walks through three progressively richer simulation modes:
1. **Real Syn3A baseline** — 458 proteins, enzymes turn over at real k_cat (no metabolite coupling)
2. **Priority 1.5 (reversibility + uptake)** — reversible Michaelis-Menten, medium uptake, steady-state metabolism
3. **Priority 2 (gene expression)** — add transcription, translation, mRNA degradation

Each produces an MP4 you can download.

Runtime → Change runtime type → **High-RAM CPU** is fine (GPU isn't used by the simulator itself).

Expected total time to run all cells: **~15-30 minutes** depending on whether you run the full Priority 2 (slowest).

## 1. Clone your fork and the upstream data

In [ ]:
# Clone the repo. Code lives in cell_sim/ inside Nikku03/cell.
import os

REPO_OWNER = "Nikku03"
REPO_NAME  = "cell"         # change if you forked to a different name
SUBDIR     = "cell_sim"     # subfolder inside the repo containing the simulator

REPO_ROOT = f"/content/{REPO_NAME}"
WORK_DIR  = f"{REPO_ROOT}/{SUBDIR}"

if not os.path.isdir(REPO_ROOT):
    !git clone https://github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_ROOT}

%cd $WORK_DIR

# Clone the Luthey-Schulten upstream data (real Syn3A kinetics)
if not os.path.isdir('data/Minimal_Cell_ComplexFormation'):
    !mkdir -p data
    !cd data && git clone --depth 1 https://github.com/Luthey-Schulten-Lab/Minimal_Cell_ComplexFormation.git

!ls -la data/Minimal_Cell_ComplexFormation/input_data/

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!which ffmpeg || apt-get install -y ffmpeg

## 3. Verify data loading

In [ ]:
import sys
sys.path.insert(0, WORK_DIR)  # set in the clone cell above
from pathlib import Path

from layer0_genome.syn3a_real import build_real_syn3a_cellspec
from layer3_reactions.sbml_parser import parse_sbml
from layer3_reactions.kinetics import load_all_kinetics, load_medium

spec, counts, complexes, kcats = build_real_syn3a_cellspec()
sbml = parse_sbml(Path('data/Minimal_Cell_ComplexFormation/input_data/Syn3A_updated.xml'))
kinetics = load_all_kinetics()
medium = load_medium()

print(f'\nSummary:')
print(f'  Genome: {spec.genome_size_bp:,} bp, {len(spec.proteins)} genes')
print(f'  Proteins with counts: {len(counts)} (total molecules: {sum(counts.values()):,})')
print(f'  Known complexes: {len(complexes)}')
print(f'  Metabolic SBML: {len(sbml.species)} species, {len(sbml.reactions)} reactions')
print(f'  Kinetic parameters: {len(kinetics)} reactions ({sum(1 for k in kinetics.values() if k.is_reversible)} reversible)')
print(f'  Medium: {len(medium)} external species')

## 4. Quick real-Syn3A demo (fastest, ~90s wall)

In [ ]:
!python tests/render_real_syn3a.py

In [ ]:
from IPython.display import HTML
from base64 import b64encode
mp4 = open('data/real_syn3a_movie/real_syn3a.mp4', 'rb').read()
HTML(f'<video width=800 controls src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')

## 5. Priority 1.5 — reversibility + medium uptake (recommended; ~100s wall for 1s simulated)

In [ ]:
!python tests/render_priority_15.py

In [ ]:
mp4 = open('data/priority_15_movie/priority_15.mp4', 'rb').read()
HTML(f'<video width=800 controls src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')

## 6. Priority 2 — full central dogma (slowest; ~400s+ wall for 1.5s simulated)
Adds transcription, translation, and mRNA degradation.

In [ ]:
!python tests/render_priority_2.py

In [ ]:
mp4 = open('data/priority_2_movie/priority_2.mp4', 'rb').read()
HTML(f'<video width=800 controls src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')

## 7. Download the MP4s

In [ ]:
from google.colab import files
for path in [
    'data/real_syn3a_movie/real_syn3a.mp4',
    'data/priority_15_movie/priority_15.mp4',
    'data/priority_2_movie/priority_2.mp4',
]:
    if os.path.isfile(path):
        files.download(path)
    else:
        print(f'(skipping {path} - not generated yet)')

## 8. Interactive exploration
Run a short Priority 1.5 simulation and inspect metabolite changes directly.

In [ ]:
import io, contextlib, time
import numpy as np

from layer2_field.dynamics import CellState, EventSimulator
from layer2_field.real_syn3a_rules import (
    populate_real_syn3a, make_folding_rule, make_complex_formation_rules,
)
from layer3_reactions.reversible import (
    build_reversible_catalysis_rules, initialize_medium,
)
from layer3_reactions.coupled import (
    initialize_metabolites, get_species_count, count_to_mM,
)

rev_rules, _ = build_reversible_catalysis_rules(sbml, kinetics)

state = CellState(spec)
populate_real_syn3a(state, counts, scale_factor=0.02, max_per_gene=10)
initialize_metabolites(state, sbml, cell_volume_um3=(4/3)*np.pi*0.2**3)
initialize_medium(state, medium)

rules = (
    [make_folding_rule(20.0)]
    + rev_rules
    + make_complex_formation_rules(complexes, base_rate_per_s=0.05)
)

sim = EventSimulator(state, rules, mode='gillespie', seed=42)
initial = dict(state.metabolite_counts)

print('Running 0.3s...')
t0 = time.time()
sim.run_until(t_end=0.3, max_events=500_000)
print(f'  {len(state.events):,} events in {time.time()-t0:.1f}s wall')

print('\nKey metabolite changes:')
for sid in ['M_atp_c', 'M_adp_c', 'M_g6p_c', 'M_pep_c', 'M_pyr_c',
            'M_lac__L_c', 'M_nadh_c', 'M_pi_c']:
    d = get_species_count(state, sid) - initial.get(sid, 0)
    dmM = count_to_mM(abs(d), state.metabolite_volume_L) * (1 if d >= 0 else -1)
    print(f'  {sid:14s}  delta={d:>+10,}  ({dmM:>+7.3f} mM)')

In [ ]:
# Plot fwd vs reverse event counts for top-turnover reactions
from collections import Counter
import matplotlib.pyplot as plt

cat_counts = Counter()
for e in state.events:
    if e.rule_name.startswith('catalysis:'):
        parts = e.rule_name.split(':')
        rxn = parts[1]
        is_rev = len(parts) > 2 and parts[2] == 'rev'
        cat_counts[(rxn, is_rev)] += 1

rxn_totals = Counter()
for (rxn, _), n in cat_counts.items():
    rxn_totals[rxn] += n
top_rxns = [r for r, _ in rxn_totals.most_common(12)]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(top_rxns))
fwd = [cat_counts.get((r, False), 0) for r in top_rxns]
rev = [cat_counts.get((r, True), 0) for r in top_rxns]
ax.bar(x - 0.2, fwd, 0.4, label='forward', color='#4dabf7')
ax.bar(x + 0.2, rev, 0.4, label='reverse', color='#f97316')
ax.set_xticks(x)
ax.set_xticklabels(top_rxns, rotation=45)
ax.set_ylabel('event count')
ax.set_title('Forward vs reverse flux for top-turnover reactions')
ax.legend()
plt.tight_layout()
plt.show()

## 9. What to do with an RTX 6000 Pro
On Blackwell with ~95 GB VRAM:

1. **Scale up molecules.** Remove `scale_factor=0.02` to simulate all ~160,000 real molecules.

2. **Scale up simulated time.** With real scale on GPU-accelerated Gillespie, wall time per second of biology should be ~1-2 minutes (vs ~100s/s on Colab CPU with 2.7k molecules).

3. **Wire up MACE-OFF for Layer 1** for novel-substrate k_cat estimation.

4. **Add PTS glucose transport.** See Non-Random-Binding sheet in kinetic_params.xlsx for the special rate law (mass-action cascade of 5 sub-reactions).